# Recherche Strategie #6 : Anomalie de volatilite idiosyncratique (IVOL)

**Question falsifiable** : sur un panier d'ETF sectoriels et larges (US, international), les actifs a **faible volatilite idiosyncratique** (variance residuelle d'un modele de marche) surperforment-ils ceux a **forte volatilite idiosyncratique** ? C'est l'hypothese d'Ang, Hodrick, Xing & Zhang (2006) : le risque *diversifiable* est mal remunere, donc les actifs a fort IVOL sous-performent ensuite.

**Distinction avec la Strategie #5 (Quality + LowVol)** : la #5 trie sur la volatilite **totale** en portefeuille **long-only**. Ici on trie sur la volatilite **residuelle** (idiosyncratique, nette du marche) et on teste un portefeuille **long-short market-neutral** (long faible IVOL - short fort IVOL). Signal, portefeuille et question different : la #5 mesurait le risque total, ce carnet isole la part non systemique.

**Donnees** : panier local `datasets/panier/panier_close_all.csv` (dossier gitignore, instantane jusqu au 2026-05-03). Univers = 14 ETF actions (SPY sert de facteur marche et est exclu de l univers). Aucun titre isole type FAANG/Mag7 : que des ETF diversifies.

**Methodologie** : modele de marche roulant sur 63 jours, tri mensuel, long-short, couts 10 bps, walk-forward 5-fold, 4 seeds, test de Newey-West et test de randomisation. Verdict honnete : BEATS / NO BEATS / INCONCLUSIVE.

## Navigation
- [Quality + LowVol (Strategie #5)](research_quality_lowvol.ipynb)
- [Risk Parity](research_risk_parity.ipynb)
- [Composite Momentum + Regime](research_composite_mom_regime.ipynb)


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# ---- Resolution robuste du panier : aucun chemin absolu machine ----
def find_panier():
    """Localise panier_close_all.csv en remontant depuis le repertoire courant.

    Aucun chemin absolu : le carnet reste portable entre postes. Le dossier
    datasets/ est gitignore, il se genere localement (voir le carnet de setup
    de la serie QuantConnect).
    """
    rel_a = Path("MyIA.AI.Notebooks/QuantConnect/datasets/panier/panier_close_all.csv")
    rel_b = Path("datasets/panier/panier_close_all.csv")
    here = Path.cwd()
    for base in [here, *here.parents]:
        for cand in (base / rel_a, base / rel_b):
            if cand.exists():
                return cand
    raise FileNotFoundError(
        "Panier introuvable : placez-vous a la racine du depot CoursIA. "
        "Le fichier attendu est "
        "MyIA.AI.Notebooks/QuantConnect/datasets/panier/panier_close_all.csv "
        "(dossier datasets/ gitignore, a generer localement)."
    )

PANIER = find_panier()
print("Panier :", PANIER.name)

# ---- Configuration ----
MARKET = "SPY"
UNIVERSE = ["RSP", "IWM", "XLF", "XLK", "XLV", "XLY", "XLP", "XLI",
            "XLU", "XLB", "XLRE", "XLC", "EFA", "EEM"]
IVOL_WINDOW = 63      # jours : fenetre du modele de marche pour l IVOL residuel
REBAL_FREQ = 21       # rebalancement mensuel
K_LEG = 4             # taille de chaque jambe (long faible IVOL / short fort IVOL)
TX_COST = 0.001       # 10 bps par unite de rotation
SEEDS = [0, 1, 7, 42]
N_SPLITS = 5

# ---- Chargement ----
df = pd.read_csv(PANIER, index_col=0, parse_dates=True).sort_index()
prices = df[[MARKET] + UNIVERSE].ffill()
prices = prices[prices.index.dayofweek < 5]   # jours ouvrables (le panier crypto est 365j)
prices = prices.dropna(how="all")
print("Univers actions : %d ETF ; periode %s -> %s" % (
    len(UNIVERSE), prices.index.min().date(), prices.index.max().date()))
print("Symboles :", UNIVERSE)


Panier : panier_close_all.csv
Univers actions : 14 ETF ; periode 2015-01-02 -> 2026-05-01
Symboles : ['RSP', 'IWM', 'XLF', 'XLK', 'XLV', 'XLY', 'XLP', 'XLI', 'XLU', 'XLB', 'XLRE', 'XLC', 'EFA', 'EEM']


## 1. Volatilite idiosyncratique par modele de marche

Pour chaque actif, on regresse son rendement quotidien sur celui du marche (SPY) sur une fenetre glissante de 63 jours, puis on prend l ecart-type des residus : c est l IVOL. Une valeur elevee signifie que l actif bouge pour des raisons propres (sectorielles), mal diversifiables a l interieur du marche.


In [2]:
def compute_ivol(prices, market, universe, window=63):
    """IVOL = ecart-type glissant des residus du modele de marche r_i = a + b*r_m + e.

    Le beta roulant est cov(r_i, r_m) / var(r_m) sur la fenetre ; le residu est
    r_i - (a + b*r_m). On renvoie un DataFrame date x actif.
    """
    ret = prices.pct_change()
    m = ret[market]
    ivol = {}
    for sym in universe:
        y = ret[sym]
        cov = y.rolling(window, min_periods=int(window * 0.8)).cov(m)
        var = m.rolling(window, min_periods=int(window * 0.8)).var()
        beta = cov / var
        alpha = (y.rolling(window, min_periods=int(window * 0.8)).mean()
                 - beta * m.rolling(window, min_periods=int(window * 0.8)).mean())
        resid = y - (alpha + beta * m)
        ivol[sym] = resid.rolling(window, min_periods=int(window * 0.8)).std()
    return pd.DataFrame(ivol)

ivol = compute_ivol(prices, MARKET, UNIVERSE, IVOL_WINDOW)
ivol_m = ivol.resample("ME").last()   # tri mensuel sur la derniere valeur du mois
valid = ivol_m.notna().sum(axis=1)
print("IVOL : %d mois x %d actifs" % (ivol_m.shape[0], ivol_m.shape[1]))
print("Actifs valides par mois : min=%d median=%d max=%d" % (
    valid.min(), int(valid.median()), valid.max()))
last = ivol_m.iloc[-1].dropna().sort_values()
print()
print("Classement IVOL - dernier mois (%s), faible -> fort :" % ivol_m.index[-1].date())
for sym, v in last.items():
    print("  %-6s %8.4f" % (sym, v))


IVOL : 137 mois x 14 actifs
Actifs valides par mois : min=0 median=14 max=14

Classement IVOL - dernier mois (2026-05-31), faible -> fort :
  RSP      0.0043
  IWM      0.0056
  XLC      0.0060
  XLY      0.0064
  XLK      0.0065
  EFA      0.0073
  XLF      0.0077
  XLV      0.0078
  XLI      0.0081
  XLRE     0.0081
  XLP      0.0091
  XLB      0.0092
  EEM      0.0096
  XLU      0.0106


## Exercice 1 - Stabilite de l IVOL selon la fenetre

La fenetre de 63 jours est un choix. L IVOL est-il stable si on prend 21 jours (court terme) ou 126 jours (long terme) ? Un signal robuste doit classer les actifs de facon similaire quelle que soit la fenetre raisonnable.


In [3]:
# Exercice 1 : comparer l IVOL a differentes fenetres (correlation de rang)
# TODO etudiant : ajoutez la fenetre 252 jours et commentez l effet du bruit de
#                 mesure sur le rang quand la fenetre raccourcit.
def rank_corr_last_month(win_a, win_b):
    a = compute_ivol(prices, MARKET, UNIVERSE, win_a).resample("ME").last().iloc[-1].dropna()
    b = compute_ivol(prices, MARKET, UNIVERSE, win_b).resample("ME").last().iloc[-1].dropna()
    common = a.index.intersection(b.index)
    return a[common].rank().corr(b[common].rank(), method="spearman")

print("Correlation de rang (Spearman) sur le dernier mois :")
print("  21j  vs 63j  : %+.3f" % rank_corr_last_month(21, 63))
print("  63j  vs 126j : %+.3f" % rank_corr_last_month(63, 126))


Correlation de rang (Spearman) sur le dernier mois :


  21j  vs 63j  : +0.732
  63j  vs 126j : +0.969


## 2. Backtest long-short (faible IVOL - fort IVOL)

Chaque mois : on classe les 14 actifs par IVOL, on achete a poids egal les K plus faibles et on vend a decouvert les K plus forts. Le rendement est net de 10 bps par unite de rotation (convention de la #5). Un rendement L-S positif signifie que le faible IVOL surperforme le fort IVOL, c est-a-dire l anomalie.


In [4]:
ret = prices.pct_change()
monthly = ret.resample("ME").apply(lambda x: (1 + x).prod() - 1)   # rendements mensuels composes

def run_ivol_longshort(ivol_m, monthly, k=4, tx_cost=0.001):
    """Backtest mensuel long faible IVOL (bottom-k) - short fort IVOL (top-k), net de couts."""
    rows = []
    prev_long = prev_short = None
    for i in range(len(ivol_m) - 1):
        sig = ivol_m.index[i]
        fut = monthly.index[monthly.index > sig]
        if len(fut) == 0:
            continue
        rd = fut[0]
        row = ivol_m.loc[sig].dropna()
        if len(row) < 2 * k:
            continue
        s = row.sort_values()
        long_low = list(s.index[:k])       # faible IVOL
        short_high = list(s.index[-k:])    # fort IVOL
        nr = monthly.loc[rd]
        lg = nr.reindex(long_low)
        sg = nr.reindex(short_high)
        if lg.isna().any() or sg.isna().any():
            continue
        lr = lg.mean()
        sr = sg.mean()
        ls = lr - sr
        if prev_long is not None:
            turn = (len(set(long_low) - set(prev_long)) + len(set(short_high) - set(prev_short))) / (2.0 * k)
            ls -= turn * tx_cost
        else:
            ls -= tx_cost
        rows.append({"date": rd, "long_low_ivol": lr, "short_high_ivol": sr, "ls": ls})
        prev_long, prev_short = long_low, short_high
    return pd.DataFrame(rows).set_index("date")

def sharpe_ann(r):
    r = r.dropna()
    if len(r) < 3 or r.std() == 0:
        return float("nan")
    return r.mean() / r.std() * np.sqrt(12)

def cagr(r):
    r = r.dropna()
    if len(r) < 2:
        return float("nan")
    return (1 + r).prod() ** (12.0 / len(r)) - 1

bt = run_ivol_longshort(ivol_m, monthly, K_LEG, TX_COST)
ls_sh = sharpe_ann(bt["ls"])
ls_cg = cagr(bt["ls"])
print("Backtest long-short IVOL : %d mois (%s -> %s)" % (
    len(bt), bt.index.min().date(), bt.index.max().date()))
print("  Sharpe annualise (L-S net) : %+.3f" % ls_sh)
print("  CAGR (L-S net)             : %+.2f%%" % (100 * ls_cg))
print("  Rendement moyen mensuel    : %+.3f%%" % (100 * bt["ls"].mean()))


Backtest long-short IVOL : 132 mois (2015-06-30 -> 2026-05-31)
  Sharpe annualise (L-S net) : +0.183
  CAGR (L-S net)             : +1.16%
  Rendement moyen mensuel    : +0.123%


## 3. Walk-forward 5-fold + multi-seed

Validation robuste : walk-forward 5-fold (expanding) sur les rendements L-S, puis 4 seeds avec un jitter multiplicatif de 10 % sur l IVOL pour tester la robustesse du classement au bruit de mesure.


In [5]:
n = len(bt)
fold = n // (N_SPLITS + 1)
wf = []
for kk in range(1, N_SPLITS + 1):
    a = fold * kk
    b = min(a + fold, n)
    if b - a < 6:
        continue
    seg = bt.iloc[a:b]["ls"]
    wf.append({"fold": kk, "n_oos": len(seg), "ls_sharpe": sharpe_ann(seg), "ls_mean": seg.mean()})
wf_df = pd.DataFrame(wf)
print("Walk-forward 5-fold (Sharpe L-S hors-echantillon) :")
print(wf_df.round(4).to_string(index=False))

print()
seed_sh = []
for seed in SEEDS:
    rng = np.random.default_rng(seed)
    noisy = ivol_m.copy()
    for c in noisy.columns:
        noisy[c] = noisy[c] * (1 + rng.normal(0, 0.10, len(noisy)))
    bts = run_ivol_longshort(noisy, monthly, K_LEG, TX_COST)
    seed_sh.append(sharpe_ann(bts["ls"]))
seed_sh = np.array(seed_sh, dtype=float)
print("Multi-seed Sharpe L-S :", np.round(seed_sh, 3).tolist())
print("  moyenne %+.3f   ecart-type %.3f" % (np.nanmean(seed_sh), np.nanstd(seed_sh)))


Walk-forward 5-fold (Sharpe L-S hors-echantillon) :
 fold  n_oos  ls_sharpe  ls_mean
    1     22     0.2492   0.0016
    2     22     0.7980   0.0055
    3     22    -0.3271  -0.0017
    4     22     0.0195   0.0002
    5     22    -0.0490  -0.0003



Multi-seed Sharpe L-S : [0.031, 0.413, 0.381, 0.314]
  moyenne +0.285   ecart-type 0.151


## 4. Analyse par regime de marche

L anomalie IVOL depend-elle du regime ? On segmente les mois selon le momentum 6 mois de SPY (haussier si positif, baissier sinon) et on mesure la performance L-S dans chaque regime.


In [6]:
spy_m = prices[MARKET].resample("ME").last()
mom6 = spy_m.pct_change(6)
regime = pd.Series(index=spy_m.index, dtype=object)
regime[mom6 >= 0] = "haussier"
regime[mom6 < 0] = "baissier"
bt_r = bt.copy()
bt_r["regime"] = regime.reindex(bt_r.index).ffill()
print("Performance L-S par regime (Sharpe, moyenne mensuelle) :")
for rg in ["haussier", "baissier"]:
    seg = bt_r[bt_r["regime"] == rg]["ls"]
    if len(seg) >= 3:
        print("  %-10s n=%3d  Sharpe %+.3f  moy %+.3f%%" % (rg, len(seg), sharpe_ann(seg), 100 * seg.mean()))


Performance L-S par regime (Sharpe, moyenne mensuelle) :
  haussier   n=104  Sharpe +0.020  moy +0.012%
  baissier   n= 27  Sharpe +0.570  moy +0.479%


## 5. Test statistique - Newey-West et randomisation

Deux garde-fous contre le data-mining. (1) t-stat de Newey-West : il corrige l autocorrelation des rendements L-S, qu un t-stat naif surestime. (2) Test de randomisation : on permute le classement IVOL dans chaque mois N fois et on mesure a quelle frequence un Sharpe L-S au moins aussi bon apparait par pur hasard (p-valeur).


In [7]:
import statsmodels.api as sm

def newey_west_t(series, lags=6):
    s = series.dropna()
    if len(s) < 10:
        return float("nan"), float("nan")
    X = np.ones(len(s))
    res = sm.OLS(s.values, X).fit(cov_type="HAC", cov_kwds={"maxlags": lags})
    return float(res.tvalues[0]), float(res.pvalues[0])

nw_t, nw_p = newey_west_t(bt["ls"])
print("Newey-West t-stat (HAC, lags=6) : t=%+.3f  p=%.4f" % (nw_t, nw_p))

def _perm_row(row, rng):
    vals = row.values.astype(float)
    mask = ~np.isnan(vals)
    out = vals.copy()
    out[mask] = rng.permutation(vals[mask])
    return pd.Series(out, index=row.index)

def random_ls_sharpe(ivol_m, monthly, k, tx_cost, rng):
    perm = ivol_m.apply(lambda r: _perm_row(r, rng), axis=1)
    b = run_ivol_longshort(perm, monthly, k, tx_cost)
    return sharpe_ann(b["ls"])

rng = np.random.default_rng(12345)
obs = sharpe_ann(bt["ls"])
N_PERM = 100
perm_sh = np.array([random_ls_sharpe(ivol_m, monthly, K_LEG, TX_COST, rng) for _ in range(N_PERM)], dtype=float)
perm_sh = perm_sh[~np.isnan(perm_sh)]
p_perm = (np.sum(perm_sh >= obs) + 1) / (len(perm_sh) + 1)
p5, p50, p95 = np.percentile(perm_sh, [5, 50, 95])
print("Randomisation (%d permutations) : Sharpe L-S observe %+.3f" % (len(perm_sh), obs))
print("  percentiles 5/50/95 : %+.3f / %+.3f / %+.3f" % (p5, p50, p95))
print("  p-valeur (>= observe) : %.4f" % p_perm)


Newey-West t-stat (HAC, lags=6) : t=+0.763  p=0.4454


Randomisation (100 permutations) : Sharpe L-S observe +0.183
  percentiles 5/50/95 : -0.569 / -0.064 / +0.339
  p-valeur (>= observe) : 0.1584


## 6. Verdict et conclusion

Regle de verdict : **BEATS** exige un Sharpe L-S net sensiblement positif, une majorite de folds walk-forward positifs, un edge cross-seed d au moins 2 sigma, **et** un test de randomisation significatif (p < 0.05). **NO BEATS** si le Sharpe L-S est negatif ou si moins de 2 folds sont positifs. Sinon **INCONCLUSIVE**. Aucun de ces criteres n est relache : un resultat mitige se declare comme tel.


In [8]:
mean_wf = wf_df["ls_sharpe"].mean() if len(wf_df) else float("nan")
n_pos = int((wf_df["ls_sharpe"] > 0).sum()) if len(wf_df) else 0
if seed_sh.std() > 1e-9:
    sigma_edge = np.nanmean(seed_sh) / np.nanstd(seed_sh)
else:
    sigma_edge = float("nan")

print("=" * 70)
print("VERDICT - Strategie #6 : Anomalie de volatilite idiosyncratique (IVOL)")
print("=" * 70)
print("Sharpe L-S (net)    : %+.3f" % ls_sh)
print("Walk-forward moyen  : %+.3f (%d/%d folds positifs)" % (mean_wf, n_pos, N_SPLITS))
if np.isnan(sigma_edge):
    print("Edge cross-seed     : N/A")
else:
    print("Edge cross-seed     : %+.2f sigma" % sigma_edge)
print("Newey-West t / p    : %+.3f / %.4f" % (nw_t, nw_p))
print("Randomisation p     : %.4f" % p_perm)

if ls_sh > 0.1 and n_pos >= 3 and (np.isnan(sigma_edge) or sigma_edge >= 2.0) and p_perm < 0.05:
    verdict = "BEATS"
elif ls_sh < 0 or n_pos < 2:
    verdict = "NO BEATS"
else:
    verdict = "INCONCLUSIVE"
print()
print(">>> VERDICT : %s <<<" % verdict)

print()
print("=" * 70)
print("Scoreboard #35 - 6 strategies")
print("=" * 70)
scoreboard = [
    (1, "Dual Momentum (L2)", "NO BEATS", "#1575"),
    (2, "Risk Parity", "NO BEATS", "#1578"),
    (3, "Trend Following (L1)", "NO BEATS", "#1574"),
    (4, "VRP Put-Write", "NO BEATS", "#1579"),
    (5, "Quality + LowVol", "NO BEATS", "research_quality_lowvol.ipynb"),
    (6, "IVOL Anomaly", verdict, "Ce carnet"),
]
for n_, name, v, pr in scoreboard:
    mark = " <<<" if n_ == 6 else ""
    print("  %d. %-24s | %-15s | %s%s" % (n_, name, v, pr, mark))


VERDICT - Strategie #6 : Anomalie de volatilite idiosyncratique (IVOL)
Sharpe L-S (net)    : +0.183
Walk-forward moyen  : +0.138 (3/5 folds positifs)
Edge cross-seed     : +1.89 sigma
Newey-West t / p    : +0.763 / 0.4454
Randomisation p     : 0.1584

>>> VERDICT : INCONCLUSIVE <<<

Scoreboard #35 - 6 strategies
  1. Dual Momentum (L2)       | NO BEATS        | #1575
  2. Risk Parity              | NO BEATS        | #1578
  3. Trend Following (L1)     | NO BEATS        | #1574
  4. VRP Put-Write            | NO BEATS        | #1579
  5. Quality + LowVol         | NO BEATS        | research_quality_lowvol.ipynb
  6. IVOL Anomaly             | INCONCLUSIVE    | Ce carnet <<<


## 7. Exercices d extension

Deux pistes pour prolonger l analyse. Chacune s execute de bout en bout et affiche un resultat reel ; les `TODO etudiant` indiquent l extension a coder.


### Exercice 2 - IVOL baissier (downside IVOL)

L IVOL complet melange les jours haussiers et baissiers. Le risque qui compte pour un investisseur long est souvent celui des mauvais jours. On recalcule un IVOL residuel sur les seuls jours ou le marche baisse.


In [9]:
# Exercice 2 : IVOL residuel calcule sur les seuls jours de marche baissiere
# TODO etudiant : comparez ce classement baissier a l IVOL complet de la section 1,
#                 et testez si le portefeuille L-S baissier est plus performant.
ret_x = prices.pct_change()
m_x = ret_x[MARKET]
down = m_x < 0
resid_down = {}
for sym in UNIVERSE:
    y = ret_x[sym]
    cov = y.rolling(IVOL_WINDOW, min_periods=40).cov(m_x)
    var = m_x.rolling(IVOL_WINDOW, min_periods=40).var()
    beta = cov / var
    r = y - beta * m_x
    resid_down[sym] = r.where(down).rolling(IVOL_WINDOW, min_periods=20).std()
down_ivol = pd.DataFrame(resid_down).resample("ME").last()
last_down = down_ivol.iloc[-1].dropna().sort_values()
print("IVOL baissier - dernier mois (faible -> fort) :")
for sym, v in last_down.items():
    print("  %-6s %8.4f" % (sym, v))


IVOL baissier - dernier mois (faible -> fort) :
  RSP      0.0047
  XLY      0.0047
  IWM      0.0062
  XLC      0.0063
  XLK      0.0074
  EFA      0.0079
  XLI      0.0082
  XLV      0.0085
  XLRE     0.0093
  XLF      0.0095
  XLP      0.0101
  EEM      0.0111
  XLB      0.0112
  XLU      0.0125


### Exercice 3 - Double tri IVOL x momentum

Un signal IVOL seul peut etre bruite. Un double tri (faible IVOL **et** fort momentum) cherche des actifs a la fois peu idiosyncratiques et en tendance haussiere.


In [10]:
# Exercice 3 : double tri faible IVOL x fort momentum 12-1
# TODO etudiant : testez d autres horizons de momentum et un tri independant a 3 groupes
#                 (faible IVOL uniquement, fort momentum uniquement, combinaison).
mom12 = prices.resample("ME").last().pct_change(12)
last_ivol = ivol_m.iloc[-1].dropna()
last_mom = mom12.iloc[-1].reindex(last_ivol.index).dropna()
common = last_ivol.index.intersection(last_mom.index)
dfx = pd.DataFrame({"ivol": last_ivol[common], "mom": last_mom[common]})
dfx["ivol_rank"] = dfx["ivol"].rank()
dfx["mom_rank"] = dfx["mom"].rank()
dfx["score"] = (len(dfx) - dfx["ivol_rank"]) + dfx["mom_rank"]   # faible IVOL + fort momentum
top = dfx.sort_values("score", ascending=False).head(K_LEG)
print("Double tri IVOL x momentum - derniers candidats (faible IVOL + fort mom) :")
for sym, r in top.iterrows():
    print("  %-6s IVOL=%.4f  mom12=%+.3f" % (sym, r["ivol"], r["mom"]))


Double tri IVOL x momentum - derniers candidats (faible IVOL + fort mom) :
  IWM    IVOL=0.0056  mom12=+0.377
  XLK    IVOL=0.0065  mom12=+0.410
  RSP    IVOL=0.0043  mom12=+0.169
  EFA    IVOL=0.0073  mom12=+0.191
